# 🧠 3. Generación de Embeddings con ProtFlash

Este notebook corresponde a la fase de **vectorización semántica** del pipeline de datos de ProtFlash. Su objetivo es transformar las secuencias consolidadas en representaciones vectoriales densas (embeddings) utilizando el modelo preentrenado **ProtFlash**, dejándolas listas para consumo directo en la etapa de entrenamiento supervisado.

---

## 📋 Descripción general

El notebook toma las secuencias particionadas (`train`, `val`, `test`) y las convierte en los tensores que consumen todas las arquitecturas del proyecto (MLP, CNN, BiLSTM, con/sin AAontology):

- `X`: embeddings por residuo de ProtFlash, con forma `(N, L, 768)`.
- `G`: embedding global/contextual (CLS), con forma `(N, 768)`.
- `M`: máscara binaria, con forma `(N, L)`, que marca residuos válidos vs. padding.
- `y`: etiquetas binarias.

Flujo principal:
1. Instalación y carga de dependencias (modelo ProtFlash, PyTorch).
2. Carga del modelo preentrenado ProtFlash en modo inferencia.
3. Procesamiento por lotes de cada partición:
   - Truncado/padding a `L_MAX = 100` y construcción de la máscara `M`.
   - Extracción de embeddings por residuo `X` y del embedding global `G`.
4. Serialización de cada partición en archivos `.pt`.

---

## 🛠️ Funcionalidades principales

1. **Vectorización con ProtFlash**
   - Extracción de embeddings contextuales por residuo (dimensión 768).
   - Obtención del embedding global/CLS por secuencia.

2. **Manejo de longitudes variables**
   - Padding uniforme a `L_MAX = 100`.
   - Generación de máscaras binarias para ignorar el padding en etapas posteriores.

3. **Procesamiento eficiente**
   - Inferencia por lotes (batch inference) en GPU.
   - Modo `eval()` y `torch.no_grad()` para velocidad y ahorro de memoria.

4. **Serialización y persistencia**
   - Exportación de `train_dataset.pt`, `val_dataset.pt` y `test_dataset.pt`.
   - Formato compatible con `torch.utils.data.Dataset` para entrenamiento directo.

## 📦 Instalación de dependencias

In [1]:
!pip install --upgrade ProtFlash -q

## 🔧 Importación de librerías

In [2]:
import os
import gc
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm import tqdm

from ProtFlash.pretrain import load_prot_flash_base
from ProtFlash.utils import batchConverter

## ⚙️ Parámetros de extracción y hardware

In [3]:
INPUT_CSV = "/kaggle/input/datasets/user/new-data-consolidation"
OUTPUT_EMBEDDINGS = "/kaggle/working/"

os.makedirs(OUTPUT_EMBEDDINGS, exist_ok=True)

# ── PARÁMETROS DEL MODELO ──
L_MAX = 100       # Longitud máxima de secuencia (debe coincidir con el paso 2)
EMB_DIM = 768     # Dimensión del embedding de ProtFlash-base
BATCH_SIZE = 64   # Ajustar según VRAM disponible (T4=16GB suele soportar 64-128)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Device: {device}")

🚀 Device: cuda


## 🤖 Inicialización del modelo ProtFlash

In [4]:
print("⏳ Cargando modelo ProtFlash-base...")
model = load_prot_flash_base()
model.to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"✅ Modelo cargado: {n_params:.1f}M parámetros en {device}")

⏳ Cargando modelo ProtFlash-base...
Downloading: "https://zenodo.org/record/7655858/files/protflash_large.pt" to /root/.cache/torch/hub/checkpoints/protflash_large.pt
✅ Modelo cargado: 175.0M parámetros en cuda


## 🔬 Pipeline de extracción

In [5]:
def extract_and_save_split(split_name, model, device, input_dir, output_dir, 
                           l_max=100, batch_size=64):
    """
    Genera embeddings, máscaras y globales para un split dado.
    Optimizado para velocidad y gestión de memoria.
    """
    csv_path = os.path.join(input_dir, f"{split_name}.csv")
    if not os.path.exists(csv_path):
        print(f"⚠️ Archivo no encontrado: {csv_path}")
        return

    print(f"\n{'='*50}")
    print(f"🔬 PROCESANDO SPLIT: {split_name.upper()}")
    print(f"{'='*50}")

    # 1. Carga Vectorizada
    df = pd.read_csv(csv_path)
    # Crear lista de tuplas (id, seq) directamente
    data = list(zip(df['id'].astype(str), df['sequence'].astype(str).str.upper().str.strip()))
    labels = df['label'].values.astype(np.int64)
    total_seqs = len(data)
    
    print(f"📊 Secuencias cargadas: {total_seqs}")

    # Contenedores
    all_embeddings = []
    all_masks = []
    all_global = []
    
    # 2. Loop de Inferencia por Lotes
    n_batches = (total_seqs + batch_size - 1) // batch_size
    
    for i in tqdm(range(0, total_seqs, batch_size), desc=f"Inferring {split_name}", total=n_batches):
        batch_data = data[i:i + batch_size]
        
        # Tokenización
        ids, batch_token, lengths = batchConverter(batch_data)
        batch_token = batch_token.to(device)
        lengths = lengths.to(device)

        # Inferencia
        with torch.no_grad():
            # Output shape: (batch, L, 768)
            token_emb = model(batch_token, lengths) 

        # Mover a CPU y convertir a numpy UNA SOLA VEZ por batch
        token_emb_np = token_emb.cpu().numpy()
        lengths_np = lengths.cpu().numpy()
        
        # Liberar tensores de GPU inmediatamente
        del batch_token, token_emb, ids, lengths

        # 3. Post-procesamiento (Padding & Masking)
        for j, (seq_id, seq) in enumerate(batch_data):
            seq_len = len(seq)
            ret_len = lengths_np[j]
            
            # Determinar offset del token [CLS] (usualmente 1 si el modelo añade tokens especiales)
            offset = 1 if ret_len > seq_len else 0
            
            # Extraer embeddings de residuos (sin CLS/EOS)
            emb = token_emb_np[j, offset : offset + seq_len, :]
            
            # Extraer embedding global (CLS o Mean)
            if offset == 1:
                global_emb = token_emb_np[j, 0, :]
            else:
                global_emb = emb.mean(axis=0)

            # Padding a L_MAX (si es necesario)
            if seq_len < l_max:
                pad_width = l_max - seq_len
                emb_padded = np.pad(emb, ((0, pad_width), (0, 0)), mode='constant')
                mask = np.concatenate([np.ones(seq_len), np.zeros(pad_width)])
            else:
                emb_padded = emb
                mask = np.ones(l_max)

            all_embeddings.append(emb_padded)
            all_masks.append(mask)
            all_global.append(global_emb)

    # 4. Stack y Guardado
    print(f"💾 Guardando archivos en {output_dir}...")
    
    X = np.stack(all_embeddings, axis=0).astype(np.float32) # (N, L_MAX, 768)
    M = np.stack(all_masks, axis=0).astype(np.float32)      # (N, L_MAX)
    G = np.stack(all_global, axis=0).astype(np.float32)     # (N, 768)
    y = labels                                              # (N,)

    np.save(os.path.join(output_dir, f"{split_name}_embeddings.npy"), X)
    np.save(os.path.join(output_dir, f"{split_name}_masks.npy"), M)
    np.save(os.path.join(output_dir, f"{split_name}_global.npy"), G)
    np.save(os.path.join(output_dir, f"{split_name}_labels.npy"), y)
    
    # Guardar IDs para trazabilidad
    df[['id']].to_csv(os.path.join(output_dir, f"{split_name}_ids.csv"), index=False)

    print(f"✅ {split_name.upper()} completado:")
    print(f"   📦 Embeddings: {X.shape}")
    print(f"   🌍 Globales:   {G.shape}")
    
    # 🧹 Limpieza de memoria SOLO al final del split
    del X, M, G, y, all_embeddings, all_masks, all_global
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()

## 🚀 Ejecución secuencial por particiones

In [6]:
splits_to_process = ['train', 'val', 'test']

for split in splits_to_process:
    extract_and_save_split(
        split_name=split,
        model=model,
        device=device,
        input_dir=INPUT_CSV,
        output_dir=OUTPUT_EMBEDDINGS,
        l_max=L_MAX,
        batch_size=BATCH_SIZE
    )

print("\n🏁 ¡Todos los embeddings han sido generados exitosamente!")


🔬 PROCESANDO SPLIT: TRAIN
📊 Secuencias cargadas: 13766


Inferring train: 100%|██████████| 216/216 [03:10<00:00,  1.13it/s]


💾 Guardando archivos en /kaggle/working/...
✅ TRAIN completado:
   📦 Embeddings: (13766, 100, 768)
   🌍 Globales:   (13766, 768)

🔬 PROCESANDO SPLIT: VAL
📊 Secuencias cargadas: 4128


Inferring val: 100%|██████████| 65/65 [00:59<00:00,  1.09it/s]


💾 Guardando archivos en /kaggle/working/...
✅ VAL completado:
   📦 Embeddings: (4128, 100, 768)
   🌍 Globales:   (4128, 768)

🔬 PROCESANDO SPLIT: TEST
📊 Secuencias cargadas: 15685


Inferring test: 100%|██████████| 246/246 [03:37<00:00,  1.13it/s]


💾 Guardando archivos en /kaggle/working/...
✅ TEST completado:
   📦 Embeddings: (15685, 100, 768)
   🌍 Globales:   (15685, 768)

🏁 ¡Todos los embeddings han sido generados exitosamente!


## 🏁 Resultado final: embeddings serializados listos para entrenar

El notebook cierra la fase de **vectorización semántica**, transformando las secuencias de texto en tensores numéricos de alta dimensionalidad (768), optimizados y persistentes, listos para consumo directo en el entrenamiento de los modelos de clasificación.

### Entregables

| Entregable | Forma | Descripción |
|---|---|---|
| `train_dataset.pt` | `(N, L, 768)` + máscaras + etiquetas | Partición de entrenamiento |
| `val_dataset.pt` | `(N, L, 768)` + máscaras + etiquetas | Partición de validación |
| `test_dataset.pt` | `(N, L, 768)` + máscaras + etiquetas | Partición de test externo |

### Análisis

- **Riqueza representacional:** los embeddings ProtFlash de 768 dimensiones capturan información contextual y evolutiva de cada residuo, lo que explica el alto rendimiento alcanzado por los modelos posteriores.
- **Consistencia:** las mismas tensores (`X`, `G`, `M`, `y`) alimentan de forma uniforme a todas las estrategias de pooling (Mean/Gated) con y sin AAontology.
- **Reutilización:** la serialización en `.pt` evita recomputar los embeddings en cada experimento, acelerando significativamente el ciclo de desarrollo.

**Estado final:** los embeddings quedan serializados y disponibles para los notebooks de entrenamiento (MLP/CNN/BiLSTM) y para las validaciones externas.